# Data Ingestion: From Files to Trustworthy Documents

| Field | Value |
|---|---|
| Stage | Data foundation |
| Difficulty | Beginner to intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Ingestion is a data-contract problem, not merely a file-reading problem. A useful document keeps readable content, stable identity, provenance, and validation evidence together.

## 30-Second Summary

This notebook ingests repository-owned UTF-8 text files. It contrasts a text-only baseline with a small, explicit document contract containing a stable content ID, source path, media type, encoding, and size. A controlled quality check shows which downstream requirements the contract satisfies.

## Why This Matters

Retrieval failures often begin before chunking or embeddings: a file is decoded incorrectly, duplicated, detached from its source, or accepted while empty. If ingestion drops identity and provenance, later citations, updates, deletions, and debugging become guesswork.

## Scope

| Covers | Does not cover |
|---|---|
| Local text discovery, decoding, stable IDs, metadata, validation, bounded previews | OCR, layout extraction, deep chunking, remote connectors, access-control implementation |


## Mental Model

```text
source bytes -> discover -> decode -> normalize -> validate -> Document contract
                                                        |-> content
                                                        |-> stable ID
                                                        |-> provenance metadata
                                                        `-> quality evidence
```

The source file remains the authority. The ingested document is a traceable representation of that source, not an anonymous string. Chunking comes later and should inherit the document ID and source metadata.


In [1]:
from dataclasses import dataclass
from hashlib import sha256
from pathlib import Path
from typing import Any


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


REPO_ROOT = find_repo_root()
TEXT_DIR = REPO_ROOT / "05-DataIngestParsing/data/text_files"
SOURCE_FILES = sorted(TEXT_DIR.glob("*.txt"))

assert SOURCE_FILES, f"No text fixtures found under {TEXT_DIR}"
[path.relative_to(REPO_ROOT).as_posix() for path in SOURCE_FILES]


['05-DataIngestParsing/data/text_files/machine_learning.txt',
 '05-DataIngestParsing/data/text_files/python_intro.txt']

## How It Works

1. **Discover** only the intended file type in a bounded directory.
2. **Decode** with an explicit encoding; do not depend on a machine default.
3. **Normalize** line endings and surrounding whitespace without rewriting meaning.
4. **Identify** content with a deterministic hash so identical bytes can be recognized.
5. **Describe** provenance using repository-relative paths and a declared media type.
6. **Validate** required fields before indexing.

The content hash is useful for deduplication, but it is not a complete business identifier. Production systems often combine a source-system ID, version, and content checksum.


## Baseline

The baseline reads each file into a string. It proves that bytes can be decoded, but it throws away the source-to-document relationship. Once strings are mixed together, a retriever cannot reliably cite, update, or delete their origins.


In [2]:
baseline_documents = [path.read_text(encoding="utf-8") for path in SOURCE_FILES]
baseline_preview = [
    {"characters": len(text), "preview": text[:70].replace("\n", " ") + "..."}
    for text in baseline_documents
]
baseline_preview


[{'characters': 575,
  'preview': 'Machine Learning Basics  Machine learning is a subset of artificial in...'},
 {'characters': 489,
  'preview': 'Python Programming Introduction  Python is a high-level, interpreted p...'}]

## Technique Implementation

`IngestedDocument` is deliberately small. Its fields are portable across frameworks, while `metadata` can grow with source-specific facts. Repository-relative paths make saved outputs reproducible and avoid leaking a developer's home directory.


In [3]:
@dataclass(frozen=True)
class IngestedDocument:
    id: str
    content: str
    metadata: dict[str, Any]


def ingest_text(path: Path) -> IngestedDocument:
    raw = path.read_bytes()
    text = raw.decode("utf-8").replace("\r\n", "\n").strip()
    relative_source = path.resolve().relative_to(REPO_ROOT).as_posix()
    return IngestedDocument(
        id=f"sha256:{sha256(raw).hexdigest()}",
        content=text,
        metadata={
            "source": relative_source,
            "media_type": "text/plain",
            "encoding": "utf-8",
            "bytes": len(raw),
        },
    )


documents = [ingest_text(path) for path in SOURCE_FILES]
[
    {"id": doc.id[:19] + "...", **doc.metadata, "preview": doc.content[:55] + "..."}
    for doc in documents
]


[{'id': 'sha256:cb51c140fac4...',
  'source': '05-DataIngestParsing/data/text_files/machine_learning.txt',
  'media_type': 'text/plain',
  'encoding': 'utf-8',
  'bytes': 589,
  'preview': 'Machine Learning Basics\n\nMachine learning is a subset o...'},
 {'id': 'sha256:089319abe23c...',
  'source': '05-DataIngestParsing/data/text_files/python_intro.txt',
  'media_type': 'text/plain',
  'encoding': 'utf-8',
  'bytes': 501,
  'preview': 'Python Programming Introduction\n\nPython is a high-level...'}]

## Controlled Experiment

Both approaches read the same two files with the same UTF-8 decoder. We score each representation against four downstream requirements: non-empty content, stable identity, source provenance, and explicit decoding metadata. This is a contract-coverage check, not a retrieval-quality metric.


In [4]:
QUALITY_FIELDS = ("content", "stable_id", "source", "encoding")


def baseline_quality(text: str) -> dict[str, bool]:
    return {
        "content": bool(text.strip()),
        "stable_id": False,
        "source": False,
        "encoding": False,
    }


def contract_quality(document: IngestedDocument) -> dict[str, bool]:
    return {
        "content": bool(document.content),
        "stable_id": document.id.startswith("sha256:"),
        "source": bool(document.metadata.get("source")),
        "encoding": document.metadata.get("encoding") == "utf-8",
    }


quality_results = {
    "text_only": sum(sum(baseline_quality(text).values()) for text in baseline_documents)
    / (len(baseline_documents) * len(QUALITY_FIELDS)),
    "document_contract": sum(sum(contract_quality(doc).values()) for doc in documents)
    / (len(documents) * len(QUALITY_FIELDS)),
}
quality_results


{'text_only': 0.25, 'document_contract': 1.0}

## Evaluation

The text-only baseline covers content but none of the operational fields, so it scores **0.25**. The explicit contract covers all four requirements and scores **1.00** for these fixtures. This does not prove the parser handles arbitrary encodings, binary files, or malformed input; it proves the documented local contract.

| Check | Text only | Document contract |
|---|---:|---:|
| Non-empty content | Yes | Yes |
| Stable ID | No | Yes |
| Source provenance | No | Yes |
| Encoding recorded | No | Yes |


In [5]:
assert quality_results == {"text_only": 0.25, "document_contract": 1.0}
assert len({document.id for document in documents}) == len(documents)
assert all(not document.metadata["source"].startswith("/") for document in documents)
assert all(document.metadata["bytes"] > 0 for document in documents)
print(f"Ingestion checks passed for {len(documents)} documents.")


Ingestion checks passed for 2 documents.


## Decision Guide

| Source | Preferred ingestion path | Important evidence |
|---|---|---|
| Plain text under your control | Explicit local reader | Encoding, checksum, source path |
| Many files with common rules | Directory/connector loader plus your validation wrapper | Per-file failures and counts |
| PDF or Word | Format-aware parser | Pages/sections, tables, extraction method |
| Database | Bounded query or snapshot | Query/version, row grain, primary key |
| Remote knowledge system | Authorized connector | Source ID, permissions, modified time |

Use a framework loader when it saves format-specific work, but normalize its output into one repository-owned contract before downstream processing.


## Failure Modes and Debugging

| Symptom | Likely cause | Verify | Fix |
|---|---|---|---|
| Garbled characters | Wrong decoder | Compare bytes and declared encoding | Detect or configure encoding; quarantine uncertain files |
| Duplicate search results | Same source ingested more than once | Compare content hashes and source IDs | Deduplicate and make ingestion idempotent |
| Citation cannot be resolved | Source metadata was dropped | Trace one result back to its file | Enforce required provenance fields |
| Empty documents reach the index | Parser returned success without content | Count empty/short outputs | Reject, alert, and preserve the failure reason |
| Updates create stale copies | IDs or versions are unstable | Re-ingest one changed source | Define deterministic IDs and replacement semantics |


## Production Notes

### Observability
Record discovered, accepted, rejected, duplicate, and changed counts; parser version; duration; and failure category. Log IDs and safe metadata rather than full sensitive content.

### Safety and Guardrails
Treat every source as untrusted input. Bound file size, validate type, scan where required, preserve access-control metadata, and never let a loader silently broaden its authorized scope.

### Latency and Cost
Hashing and parsing are usually linear in input size. Incremental ingestion should skip unchanged sources and isolate retries so one bad file does not restart an entire collection.


## Practice

Add a third UTF-8 fixture with Windows line endings, ingest it twice, and verify that your chosen identity policy behaves as intended. Then add an empty file and decide whether to reject or retain it with an explicit status.

## Recall

Toggle - Recall: Why is a string not yet a trustworthy document?
It lacks stable identity, provenance, and validation evidence needed for citations and lifecycle operations.

Toggle - Recall: What does a content hash prove?
It identifies the exact bytes hashed; it does not prove source ownership, permission, freshness, or semantic equivalence.

Toggle - Recall: Why use repository-relative source paths?
They remain portable and avoid exposing machine-specific home directories.

Toggle - Recall: Where should chunking occur?
After source-level ingestion and validation, with document identity and provenance inherited by every chunk.

## Sources

- [Python documentation: Reading and writing files](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files)
- [Python documentation: `hashlib`](https://docs.python.org/3/library/hashlib.html)
- Repository-owned fixtures under `05-DataIngestParsing/data/text_files/`

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for local UTF-8 ingestion | Add source-version and rejection fixtures |
